In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
# from langchain_milvus import Milvus
from preprocess import preprocess_text

tqdm.pandas()

DATA_DIR = Path("../../data")

In [ ]:
df = pd.read_csv(DATA_DIR/"MetaHate.tsv", sep='\t')
index_removed = pd.read_csv("excluded.csv", header=None)[0].tolist()
df = df.loc[df.index.difference(index_removed)]
df['text'] = df.progress_apply(lambda x : preprocess_text(x[1]), axis=1).tolist()

  0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_111202/2081274648.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df['text'] = df.progress_apply(lambda x : preprocess_text(x[1]), axis=1).tolist()
100%|██████████| 1000/1000 [00:01<00:00, 749.33it/s]


In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings_func = HuggingFaceEmbeddings(model_name=model_name, cache_folder="../cache", model_kwargs={'device': 'cuda'})
db = Chroma(persist_directory="search_index", embedding_function=embeddings_func, collection_metadata={"hnsw:space": "cosine"})
# db = Milvus(
#     embedding_function=embeddings_func,
#     connection_args={"uri": './milvus.db'},
#     auto_id=True,
#     index_params={"index_type": "FLAT", "metric_type": "COSINE"},
# )

/tmp/ipykernel_111202/1407458836.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings_func = HuggingFaceEmbeddings(model_name=model_name, cache_folder="../cache", model_kwargs={'device': 'cuda'})
2025-02-25 23:43:18.851876: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-25 23:43:20.986097: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libn

In [ ]:
items = df#[:1000]
batch_size = 64
for ind in tqdm(range(0, items.shape[0], batch_size)):
    slice_ = items.iloc[ind:ind+batch_size]
    ids = list(map(str, slice_.index))
    db.add_texts(
        slice_['text'].tolist(),  
        metadatas=[{'item_id': id, 'label': label} for id, label in zip(ids, slice_['label'].tolist())]
    )
print("Vectorization finished")

  0%|          | 0/63 [00:00<?, ?it/s]

['714986', '714987', '714988', '714989', '714990', '714991', '714992', '714993', '714994', '714995', '714996', '714997', '714998', '714999', '715000', '715001']
(16, 2) 16 16


  2%|▏         | 1/63 [00:02<02:04,  2.00s/it]

['715002', '715003', '715004', '715005', '715006', '715007', '715008', '715009', '715010', '715011', '715012', '715013', '715014', '715015', '715016', '715017']
(16, 2) 16 16


  5%|▍         | 3/63 [00:02<00:35,  1.69it/s]

...

['715914', '715915', '715916', '715917', '715918', '715919', '715920', '715921', '715922', '715923', '715924', '715925', '715926', '715927', '715928', '715929']
(16, 2) 16 16
['715930', '715931', '715932', '715933', '715934', '715935', '715936', '715937', '715938', '715939', '715940', '715941', '715942', '715943', '715944', '715945']
(16, 2) 16 16


 97%|█████████▋| 61/63 [00:11<00:00,  6.38it/s]

['715946', '715947', '715948', '715949', '715950', '715951', '715952', '715953', '715954', '715955', '715956', '715957', '715958', '715959', '715960', '715961']
(16, 2) 16 16
['715962', '715963', '715964', '715965', '715966', '715967', '715968', '715969', '715970', '715971', '715972', '715973', '715974', '715975', '715976', '715977']
(16, 2) 16 16


100%|██████████| 63/63 [00:11<00:00,  5.42it/s]

['715978', '715979', '715980', '715981', '715982', '715983', '715984', '715985']
(8, 2) 8 8
Vectorization finished


In [5]:
for ind, row in tqdm(items[:10].iterrows(), total=items[:10].shape[0]):
    print(ind, db.similarity_search_with_score(row['text']))

  0%|                                                                                                                                                                 | 0/10 [00:00<?, ?it/s]

0 [(Document(metadata={'item_id': '0', 'label': 0}, page_content="!! RT As a woman you shouldn't complain about cleaning up your house. & as a man you should always take the trash out.."), 3.5762786865234375e-07), (Document(metadata={'item_id': '583568', 'label': 1}, page_content="i can assure you, it is a woman's job to clean up the filth in their house"), 0.25686269998550415), (Document(metadata={'item_id': '604895', 'label': 1}, page_content="Housework is a woman's job"), 0.2674013376235962), (Document(metadata={'item_id': '583567', 'label': 0}, page_content='i can assure you, most of us women hate to clean up the filth in their houses'), 0.3131389617919922)]


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 57.43it/s]

1 [(Document(metadata={'item_id': '1', 'label': 0}, page_content='!! RT boy dats cold.. tyga dwn bad for cuffin dat hoe in the 1 st place!!'), -3.5762786865234375e-07), (Document(metadata={'item_id': '108005', 'label': 0}, page_content='RT A hoe never gets cold bea'), 0.3089747428894043), (Document(metadata={'item_id': '31390', 'label': 1}, page_content='RT me: HOES DONT GET COLD *me after being out side for 5 mins*'), 0.33167243003845215), (Document(metadata={'item_id': '68108', 'label': 0}, page_content="RT If she asks you for your hoodie or sweater wife her because that means she's cold and not a hoe because hoes never get cold"), 0.3662176728248596)]
2 [(Document(metadata={'item_id': '2', 'label': 0}, page_content='!! RT Dawg!! RT You ever fuck a bitch and she start to cry? You be confused as shit'), -1.1920928955078125e-07), (Document(metadata={'item_id': '13580', 'label': 0}, page_content='Okay! RT Right— bitch!! RT you mad cause YOU opened your mouth?'), 0.31052517890930176), (D

For each entry, find similar entries and store them:

In [6]:
from collections import defaultdict

threshold = 0.2
retriever = db.as_retriever(search_kwargs={'score_threshold': threshold})
similar = defaultdict(set)
for ind, row in tqdm(items.iterrows(), total=items.shape[0]):
    similar_results = db.similarity_search_with_score(row['text'])
    for doc, sim in similar_results:
        if str(ind) != doc.metadata.get('item_id'):
            if sim < threshold:
                similar[(ind, row['text'], row['label'])].add((doc.page_content, doc.metadata.get('item_id'), doc.metadata.get('label'), sim))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1101165/1101165 [4:33:25<00:00, 67.12it/s]


In [7]:
import json
import pickle
with open('results.pkl', 'wb') as f:
    pickle.dump((df, similar), f)
similar_output = {f"{k[0]} - {k[1]}": list(v) for k, v in similar.items()}
with open('similar.json', 'w', encoding='utf8') as f:
    json.dump(similar_output, f, indent=2)

Identify duplicate entries and store their indices:

In [8]:
import pickle 
with open('results.pkl', 'rb') as f:
    df, similar = pickle.load(f)

In [9]:
removed = []
checked = []
label_mismatches = 0
for k, data in tqdm(similar.items()):
    # Check for label mismatches, and if any found, remove all, including compared
    labels = set([v[2] for v in data])
    if len(labels) > 1:
        label_mismatches += 1
        removed.extend([v[1] for v in data])
        removed.append(k[0])
        continue
    # Check for clear duplicates and remove them as well
    for text, ind, label, _ in data:
        # TODO: this should not happen!
        if ind is None:
            continue
        if k[1].strip() == text.strip() and ind not in checked:
            removed.append(ind)
    checked.append(k[0])
# TODO: filtering None values should not be required!
removed = set(map(int, filter(lambda _: _, removed)))
print(len(removed), label_mismatches)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 613870/613870 [14:50<00:00, 689.66it/s]

232328 29416


In [10]:
with open('removed.csv', 'w') as f:
    f.write('\n'.join(map(str, sorted(removed))))

In [12]:
def sim_data_to_df(k, v):
    df = pd.concat([pd.DataFrame([k] * len(v)), pd.DataFrame(v)], axis=1)
    df.columns = ['index_query', 'text_query', 'label_query', 'text_retrieved', 'index_retrieved', 'label_retrieved', 'score']
    return df

df_similar = pd.concat([sim_data_to_df(k, v) for k, v in tqdm(similar.items())])
df_similar.to_csv('similar.csv', index=None)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 613870/613870 [05:21<00:00, 1906.58it/s]
